In [1]:
import os
import subprocess

# GitHub Repository and Folder Path
GITHUB_REPO = "https://github.com/spMohanty/PlantVillage-Dataset.git"
FOLDER_PATH = "raw/color"  # Change this if needed
DEST_PATH = "./dataset"

# Create directory
os.makedirs(DEST_PATH, exist_ok=True)

# Clone only the specific folder
print(f"Downloading {FOLDER_PATH} from GitHub...")
subprocess.run([
    "git", "clone", "--depth", "1", "--filter=blob:none",
    "--sparse", GITHUB_REPO, DEST_PATH
], check=True)

# Checkout only the needed folder
subprocess.run(["git", "-C", DEST_PATH, "sparse-checkout", "set", FOLDER_PATH], check=True)

print(f"Dataset downloaded successfully to {DEST_PATH}/{FOLDER_PATH}!")


Dataset downloaded successfully to ./dataset/raw/color!


In [5]:
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import StratifiedShuffleSplit

# --------------------------------------------------------------------------
# Global Settings
# --------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(42)
np.random.seed(42)

# --------------------------------------------------------------------------
# 1. Data Transformations
# --------------------------------------------------------------------------
class AlbumentationsTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, image):
        image = np.array(image)
        augmented = self.transform(image=image)
        return augmented['image']

def get_train_transforms():
    return AlbumentationsTransform(A.Compose([
        A.Resize(256, 256),
        A.RandomResizedCrop(224, 224, scale=(0.8, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=20, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ]))

def get_test_transforms():
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# --------------------------------------------------------------------------
# 2. Custom ImageFolder Class
# --------------------------------------------------------------------------
class CustomImageFolder(datasets.ImageFolder):
    def set_transform(self, transform):
        self.transform = transform

# --------------------------------------------------------------------------
# 3. Stratified Splitting (Train: 60%, Val: 20%, Test: 20%)
# --------------------------------------------------------------------------
def stratified_split(dataset, train_size=0.6, val_size=0.2, test_size=0.2, random_state=42):
    targets = np.array(dataset.targets)
    indices = np.arange(len(dataset))

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(1 - train_size), random_state=random_state)
    train_idx, temp_idx = next(sss1.split(indices, targets))

    temp_targets = targets[temp_idx]
    test_ratio = test_size / (test_size + val_size)
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
    val_idx, test_idx = next(sss2.split(temp_idx, temp_targets))

    return train_idx, temp_idx[val_idx], temp_idx[test_idx]

# --------------------------------------------------------------------------
# 4. Load Data and Create DataLoaders
# --------------------------------------------------------------------------
def get_dataloaders(dataset_path, batch_size=32):
    train_transform = get_train_transforms()
    test_transform = get_test_transforms()

    full_dataset = CustomImageFolder(dataset_path, transform=train_transform)
    train_idx, val_idx, test_idx = stratified_split(full_dataset)

    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(copy.deepcopy(full_dataset), val_idx)
    val_dataset.dataset.set_transform(test_transform)

    test_dataset = Subset(copy.deepcopy(full_dataset), test_idx)
    test_dataset.dataset.set_transform(test_transform)

    dataloaders = {
        "train": DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True),
        "val": DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        "test": DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    }

    class_names = full_dataset.classes
    print(f"Dataset loaded: {len(full_dataset)} images, {len(class_names)} classes.")
    return dataloaders, class_names

# --------------------------------------------------------------------------
# 5. Model Definition (Using Pretrained VGG16)
# --------------------------------------------------------------------------
class SegNetClassifier(nn.Module):
    def __init__(self, num_classes):
        super(SegNetClassifier, self).__init__()
        vgg16_bn = models.vgg16_bn(pretrained=True)
        self.features = vgg16_bn.features
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# --------------------------------------------------------------------------
# 6. Training Function
# --------------------------------------------------------------------------
def train_model(model, dataloaders, criterion, optimizer, num_epochs=10, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    early_stop_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_corrects = 0
            total = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_corrects += torch.sum(preds == labels.data)
                total += inputs.size(0)

            epoch_acc = running_corrects.double() / total
            print(f"{phase.capitalize()} Accuracy: {epoch_acc:.4f}")

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                early_stop_counter = 0
            elif phase == 'val':
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print("Early stopping triggered!")
                model.load_state_dict(best_model_wts)
                return model

    model.load_state_dict(best_model_wts)
    return model

# --------------------------------------------------------------------------
# 7. Test Evaluation Function
# --------------------------------------------------------------------------
def evaluate_model(model, dataloader):
    model.eval()
    running_corrects = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            running_corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)

    test_acc = running_corrects.double() / total
    return test_acc.item()

# --------------------------------------------------------------------------
# 8. Main Function
# --------------------------------------------------------------------------
if __name__ == "__main__":
    dataset_path = "./dataset/raw/color"
    batch_size = 32
    num_epochs = 10

    dataloaders, class_names = get_dataloaders(dataset_path, batch_size=batch_size)
    num_classes = len(class_names)

    model = SegNetClassifier(num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    model = train_model(model, dataloaders, criterion, optimizer, num_epochs=num_epochs)
    
    test_acc = evaluate_model(model, dataloaders["test"])
    print("\n" + "=" * 80)
    print(f"\033[1mFINAL TEST ACCURACY: {test_acc:.4f}\033[0m")
    print("=" * 80 + "\n")


Using device: cuda
Dataset loaded: 54305 images, 38 classes.


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/10
Train Accuracy: 0.8666
Val Accuracy: 0.9750

Epoch 2/10
Train Accuracy: 0.9603
Val Accuracy: 0.9651

Epoch 3/10
Train Accuracy: 0.9696
Val Accuracy: 0.9818

Epoch 4/10
Train Accuracy: 0.9742
Val Accuracy: 0.9831

Epoch 5/10
Train Accuracy: 0.9792
Val Accuracy: 0.9765

Epoch 6/10
Train Accuracy: 0.9795
Val Accuracy: 0.9861

Epoch 7/10
Train Accuracy: 0.9802
Val Accuracy: 0.9860

Epoch 8/10
Train Accuracy: 0.9835
Val Accuracy: 0.9904

Epoch 9/10
Train Accuracy: 0.9837
Val Accuracy: 0.9866

Epoch 10/10
Train Accuracy: 0.9850
Val Accuracy: 0.9793

FINAL TEST ACCURACY: 0.9900

